# Get gui 

In [1]:
import pickle
with open('emoji_dict.pickle', 'rb') as f:
    emoji_dict = pickle.load(f)

In [8]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoTokenizer
from helper_functions import BERTWithExtraFeatures 


In [ ]:
import tkinter as tk
from tkinter import messagebox
import pickle
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from helper_functions import BERTWithExtraFeatures
from huggingface_hub import hf_hub_download
# loads the emoji dictionary
with open('emoji_dict.pickle', 'rb') as f:
    emoji_dict = pickle.load(f)

# initialize models
sentiment_model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
emotion_model_name = "j-hartmann/emotion-english-distilroberta-base"

sentiment_tokenizer = AutoTokenizer.from_pretrained(sentiment_model_name)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(sentiment_model_name)
sentiment_model.eval()

emotion_tokenizer = AutoTokenizer.from_pretrained(emotion_model_name)
emotion_model = AutoModelForSequenceClassification.from_pretrained(emotion_model_name)
emotion_model.eval()

num_labels = 1196  
extra_feature_dim = 10  # 3 sentiment + 7 emotion

emoji_model = BERTWithExtraFeatures(num_labels=num_labels, extra_feature_dim=extra_feature_dim)
device = 'cpu'

# loads emoji model
model_path = hf_hub_download(repo_id="afreddy1/emoji", filename="model_w.pth")
emoji_model.load_state_dict(torch.load(model_path, map_location=device))
#emoji_model.load_state_dict(torch.load("model_w.pth", map_location=device))
emoji_model.to(device)
emoji_model.eval()

# loads tokenizer for emoji model (bert-base-uncased)
text_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Function to process the input text and get emojis
def predict_emojis():
    # Get the input text from the user input field
    text = text_entry.get()

    if not text:
        messagebox.showwarning("Input Error", "Please enter some text.")
        return
    
    # sentiment
    with torch.no_grad():
        sent_inputs = sentiment_tokenizer(text, return_tensors="pt", truncation=True)
        sent_outputs = sentiment_model(**sent_inputs)
        sentiment_probs = F.softmax(sent_outputs.logits, dim=1).squeeze().tolist()

    # Emoootion
    with torch.no_grad():
        emo_inputs = emotion_tokenizer(text, return_tensors="pt", truncation=True)
        emo_outputs = emotion_model(**emo_inputs)
        emotion_probs = F.softmax(emo_outputs.logits, dim=1).squeeze().tolist()

    # tokenizes input text for model
    encoding = text_tokenizer(text, truncation=True, padding='max_length', max_length=128, return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    #Combines extra features 
    extra_features = torch.tensor([sentiment_probs + emotion_probs], dtype=torch.float).to(device)

    #  gets emojis
    with torch.no_grad():
        logits = emoji_model(input_ids, attention_mask, extra_features)
        probs = torch.sigmoid(logits)

    # top 3 emojis
    top_k = 3
    top_indices = torch.topk(probs[0], top_k).indices.tolist()

    # Invert the emoji dictionary for easy lookup
    inverse_emoji_dict = {v: k for k, v in emoji_dict.items()}
    predicted_emojis = [inverse_emoji_dict[idx] for idx in top_indices if idx in inverse_emoji_dict]

    # displays the emojis in the result label
    result_label.config(text=" " + ", ".join(predicted_emojis))

# Initialize  GUI
root = tk.Tk()
root.title("Emoji Generator Tool")

# Set the window size
root.geometry("500x300")

# instructions on gui
instruction_label = tk.Label(root, text="Enter some text:")
instruction_label.pack(pady=10)

# text entry box for user input
text_entry = tk.Entry(root, width=40)
text_entry.pack(pady=10)

# button to trigger emoji getter
predict_button = tk.Button(root, text="Generate Emojis", command=predict_emojis)
predict_button.pack(pady=10)

# label to display the result
result_label = tk.Label(root, text=" ", font=("Helvetica", 14))
result_label.pack(pady=20)

# starts event loop
root.mainloop()
